In [1]:
from pathlib import Path
import sys

# adjust this depending on where the notebook sits
# example: notebook is in project/notebooks/experiments/
PROJECT_ROOT = Path.cwd().resolve().parents[0]   # go up 2 levels

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
from pathlib import Path
import matplotlib.pyplot as plt


import torch
# torch.set_num_threads(1)
# torch.set_num_interop_threads(1)
from torch.utils.data import DataLoader




import cv2
#cv2.setNumThreads(0)

from tqdm import tqdm
import numpy as np

from datasets.bcdata import BCDataDataset, collate_heatmap_points
from datasets.transforms import PointsToLocalizationHeatmap, PointsToCountHeatmap


from visualization import overlay_heatmap


from models.models import HybridModel
from models.losses import weighted_sigmoid_mse_from_logits, softplus_mse_from_logits, l1_count_from_density_logits

from src.debug import print_info

from src.weights_init import init_count_head_bias


torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [2]:
with open("../config.yaml", "r") as f:
    cfg = yaml.safe_load(f)


data_root = Path(cfg["h200_paths"]["data_root"])
checkpoint_dir = Path(cfg["h200_paths"]["checkpoint_dir"])

print(f"data_root: {data_root}")
print(f"checkpoint_dir: {checkpoint_dir}")

data_root: /raid/datasets/Yeldos/BCData
checkpoint_dir: checkpoints


In [3]:
loc_heatmap_generator = PointsToLocalizationHeatmap(out_hw=(160,160), in_hw=(640,640), sigma=2.0)
count_heatmap_generator = PointsToCountHeatmap(out_hw=(160,160), in_hw=(640,640), sigma=2.0)

dataset = BCDataDataset(root = data_root,
                        split="train",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)



In [4]:

train_loader = DataLoader(
    dataset,
    batch_size=16,        # choose based on GPU memory (640×640 images are large)
    shuffle=False,
    num_workers=4,       # use 0 if debugging
    pin_memory=True,     # recommended when using GPU
    drop_last=True,       # optional, useful for BatchNorm
    collate_fn = collate_heatmap_points
)


In [5]:
for img, loc_heatmap, count_heatmap, pos_pts, neg_pts in train_loader:
    print_info(img, "img")
    print_info(loc_heatmap, "loc_heatmap")
    print_info(count_heatmap, "count_heatmap")
    #print_info(pos_pts, "pos_pts")
    #print_info(neg_pts, "neg_pts")
    break


********************************************************************************
img: torch.Tensor | shape=(16, 3, 640, 640) | dtype=torch.float32 | device=cpu
********************************************************************************


********************************************************************************
loc_heatmap: torch.Tensor | shape=(16, 2, 160, 160) | dtype=torch.float32 | device=cpu
********************************************************************************


********************************************************************************
count_heatmap: torch.Tensor | shape=(16, 2, 160, 160) | dtype=torch.float32 | device=cpu
********************************************************************************



In [ ]:
# COMPARE HEATMAPS
import torch

@torch.no_grad()
def max_unit_mass_heatmap(points_xy, out_hw=(160,160), in_hw=(640,640), sigma=2.0, clip=False, device="cpu"):
    """
    points_xy: (N,2) in input pixel coords (x,y) for 640x640 (or in_hw).
    returns: (H,W) heatmap where each point contributes a unit-mass Gaussian and we take max over points.
    """
    H, W = out_hw
    hm = torch.zeros((H, W), dtype=torch.float32, device=device)
    if points_xy is None or points_xy.numel() == 0:
        return hm

    pts = points_xy.to(device=device, dtype=torch.float32).clone()

    inH, inW = in_hw
    sx = W / float(inW)
    sy = H / float(inH)
    pts[:, 0] *= sx
    pts[:, 1] *= sy

    if clip:
        pts[:, 0].clamp_(0, W - 1)
        pts[:, 1].clamp_(0, H - 1)

    yy = torch.arange(H, device=device, dtype=torch.float32).view(H, 1)
    xx = torch.arange(W, device=device, dtype=torch.float32).view(1, W)

    inv_2s2 = 1.0 / (2.0 * sigma * sigma)

    for x, y in pts:
        g = torch.exp(-((xx - x) ** 2 + (yy - y) ** 2) * inv_2s2)
        g_sum = g.sum().clamp_min(1e-12)
        g = g / g_sum  # unit-mass
        hm = torch.maximum(hm, g)

    return hm


@torch.no_grad()
def batch_points(pos_pts, neg_pts):
    """
    Handles common collate outputs:
    - list[Tensor(Ni,2)] length B
    - Tensor(B,Ni,2) (padded) -> treat rows with NaNs as invalid if present
    Returns: iterable of length B: (pos_i, neg_i) each Tensor(Ni,2)
    """
    if isinstance(pos_pts, list) and isinstance(neg_pts, list):
        return list(zip(pos_pts, neg_pts))

    # If tensors, assume shape (B, N, 2). If padded with NaNs, filter them.
    if torch.is_tensor(pos_pts) and torch.is_tensor(neg_pts):
        B = pos_pts.shape[0]
        out = []
        for b in range(B):
            p = pos_pts[b]
            n = neg_pts[b]
            if torch.isnan(p).any():
                p = p[~torch.isnan(p).any(dim=1)]
            if torch.isnan(n).any():
                n = n[~torch.isnan(n).any(dim=1)]
            out.append((p, n))
        return out

    raise TypeError("Unsupported pos_pts/neg_pts batch format.")


@torch.no_grad()
def analyze_target_difference(train_loader, out_hw=(160,160), in_hw=(640,640), sigma=2.0, device=None, eps=1e-8):
    """
    Compares:
      Hsum_mass = count_heatmap (already unit-mass summed)
      Hmax_mass = max over unit-mass Gaussians (recomputed from points)

    Reports per-class stats aggregated over dataset.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    stats = {
        "pos": {"n": 0, "f_ov": 0.0, "r1": 0.0, "q": 0.0, "a": 0.0},
        "neg": {"n": 0, "f_ov": 0.0, "r1": 0.0, "q": 0.0, "a": 0.0},
    }

    for img, loc_hm, count_hm, pos_pts, neg_pts in train_loader:
        # count_hm: (B,2,H,W) already unit-mass summed by your generator
        count_hm = count_hm.to(device=device, dtype=torch.float32)

        pairs = batch_points(pos_pts, neg_pts)
        B = len(pairs)

        for b in range(B):
            p_pts, n_pts = pairs[b]

            # Channel 0 = pos, 1 = neg (per your description)
            for cls_name, pts, c in [("pos", p_pts, 0), ("neg", n_pts, 1)]:
                Hsum = count_hm[b, c]  # (H,W), unit-mass sum; sum(Hsum) ~= N

                Hmax = max_unit_mass_heatmap(
                    pts, out_hw=out_hw, in_hw=in_hw, sigma=sigma, clip=False, device=device
                )

                D = (Hsum - Hmax).clamp_min(0.0)

                # Metrics (scale-consistent)
                # f_ov: fraction of pixels where sum differs from max meaningfully
                thr = 1e-6  # can also use thr = 1e-3 * Hsum.max()
                f_ov = (D > thr).float().mean()

                # r1: relative "overlap mass" energy
                r1 = D.abs().sum() / (Hsum.abs().sum() + eps)

                # q: integral ratio in unit-mass units (in (0,1])
                q = Hmax.sum() / (Hsum.sum() + eps)

                # a: peak amplification (>=1 if overlaps occur at peaks)
                a = Hsum.max() / (Hmax.max() + eps)

                s = stats[cls_name]
                s["n"] += 1
                s["f_ov"] += f_ov.item()
                s["r1"] += r1.item()
                s["q"]  += q.item()
                s["a"]  += a.item()

    # finalize means
    out = {}
    for cls in ["pos", "neg"]:
        n = max(stats[cls]["n"], 1)
        out[cls] = {k: (v / n if k != "n" else v) for k, v in stats[cls].items()}
    return out

In [9]:
results = analyze_target_difference(train_loader, out_hw=(160,160), in_hw=(640,640), sigma=2.0, device = 'cuda')
print(results)

{'pos': {'n': 800, 'f_ov': 0.08896244899976409, 'r1': 0.023212492733154046, 'q': 0.9767875245958567, 'a': 1.0593416064232588}, 'neg': {'n': 800, 'f_ov': 0.1984016562387478, 'r1': 0.03325618331680494, 'q': 0.9667438262701035, 'a': 1.0645628894865513}}


In [10]:
# COMPARE HEATMAPS for images with certain number of cells


import math
import torch

@torch.no_grad()
def max_unit_mass_heatmap(points_xy: torch.Tensor,
                          out_hw=(160, 160),
                          in_hw=(640, 640),
                          sigma=2.0,
                          clip=False,
                          device="cpu") -> torch.Tensor:
    """
    Returns (H,W) heatmap: max over points of unit-mass Gaussian kernels.
    points_xy: (N,2) in input pixel coords (x,y) for in_hw.
    """
    H, W = out_hw
    hm = torch.zeros((H, W), dtype=torch.float32, device=device)
    if points_xy is None or points_xy.numel() == 0:
        return hm

    pts = points_xy.to(device=device, dtype=torch.float32).clone()

    inH, inW = in_hw
    sx = W / float(inW)
    sy = H / float(inH)
    pts[:, 0] *= sx
    pts[:, 1] *= sy

    if clip:
        pts[:, 0].clamp_(0, W - 1)
        pts[:, 1].clamp_(0, H - 1)

    yy = torch.arange(H, device=device, dtype=torch.float32).view(H, 1)
    xx = torch.arange(W, device=device, dtype=torch.float32).view(1, W)
    inv_2s2 = 1.0 / (2.0 * sigma * sigma)

    for x, y in pts:
        g = torch.exp(-((xx - x) ** 2 + (yy - y) ** 2) * inv_2s2)
        g = g / g.sum().clamp_min(1e-12)   # unit-mass
        hm = torch.maximum(hm, g)

    return hm


@torch.no_grad()
def batch_points(pos_pts, neg_pts):
    """
    Supports:
      - list[Tensor(Ni,2)] length B
      - Tensor(B,N,2) padded with NaNs
    Returns: list length B of (pos_i, neg_i) tensors.
    """
    if isinstance(pos_pts, list) and isinstance(neg_pts, list):
        return list(zip(pos_pts, neg_pts))

    if torch.is_tensor(pos_pts) and torch.is_tensor(neg_pts):
        B = pos_pts.shape[0]
        out = []
        for b in range(B):
            p = pos_pts[b]
            n = neg_pts[b]
            if torch.isnan(p).any():
                p = p[~torch.isnan(p).any(dim=1)]
            if torch.isnan(n).any():
                n = n[~torch.isnan(n).any(dim=1)]
            out.append((p, n))
        return out

    raise TypeError("Unsupported pos_pts/neg_pts batch format.")


def _bin_index(n: int, bins):
    """
    bins: list of (lo, hi) inclusive ranges, where hi can be math.inf
    """
    for i, (lo, hi) in enumerate(bins):
        if lo <= n <= hi:
            return i
    return None


def _init_acc(bins):
    return {
        "bins": bins,
        "bin_n": [0 for _ in bins],
        "f_ov":  [0.0 for _ in bins],
        "r1":    [0.0 for _ in bins],
        "q":     [0.0 for _ in bins],
        "a":     [0.0 for _ in bins],
    }


@torch.no_grad()
def analyze_target_difference_binned(loader,
                                    out_hw=(160,160),
                                    in_hw=(640,640),
                                    sigma=2.0,
                                    device=None,
                                    bins=None,
                                    thr_mode="relative",   # "relative" or "absolute"
                                    thr_abs=1e-6,
                                    thr_rel=1e-3,
                                    eps=1e-12):
    """
    Stratifies target-difference metrics by GT count bins using pos_pts/neg_pts.

    Metrics per image per class:
      Hsum = count_heatmap (unit-mass sum)
      Hmax = max over unit-mass kernels (recomputed from points)
      D = clamp(Hsum - Hmax, min=0)

      f_ov = mean(D > thr)
      r1   = sum(D) / sum(Hsum)
      q    = sum(Hmax) / sum(Hsum)
      a    = max(Hsum) / max(Hmax)

    Returns dict with per-class results and per-bin means.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if bins is None:
        bins = [(0, 10), (11, 30), (31, 100), (101, math.inf)]

    acc = {"pos": _init_acc(bins), "neg": _init_acc(bins)}

    for img, loc_hm, count_hm, pos_pts, neg_pts in loader:
        # count_hm: (B,2,H,W) unit-mass sum targets
        count_hm = count_hm.to(device=device, dtype=torch.float32)
        pairs = batch_points(pos_pts, neg_pts)

        for b, (p_pts, n_pts) in enumerate(pairs):
            for cls_name, pts, c in [("pos", p_pts, 0), ("neg", n_pts, 1)]:
                N = int(pts.shape[0]) if (pts is not None and pts.numel() > 0) else 0
                bi = _bin_index(N, bins)
                if bi is None:
                    continue

                Hsum = count_hm[b, c]  # (H,W)
                Hmax = max_unit_mass_heatmap(
                    pts, out_hw=out_hw, in_hw=in_hw, sigma=sigma, clip=False, device=device
                )

                sum_Hsum = Hsum.sum().item()
                sum_Hmax = Hmax.sum().item()

                # Handle N=0 robustly
                if sum_Hsum < 1e-9 and sum_Hmax < 1e-9:
                    f_ov = 0.0
                    r1   = 0.0
                    q    = 1.0
                    a    = 1.0
                else:
                    D = (Hsum - Hmax).clamp_min(0.0)

                    if thr_mode == "relative":
                        thr = max(thr_abs, float(thr_rel * Hsum.max().item()))
                    else:
                        thr = thr_abs

                    f_ov = float((D > thr).float().mean().item())
                    r1   = float(D.sum().item() / (sum_Hsum + eps))
                    q    = float(sum_Hmax / (sum_Hsum + eps))
                    a    = float(Hsum.max().item() / (Hmax.max().item() + eps))

                A = acc[cls_name]
                A["bin_n"][bi] += 1
                A["f_ov"][bi]  += f_ov
                A["r1"][bi]    += r1
                A["q"][bi]     += q
                A["a"][bi]     += a

    # finalize: convert sums -> means
    out = {}
    for cls in ["pos", "neg"]:
        A = acc[cls]
        means = []
        for i, (lo, hi) in enumerate(A["bins"]):
            n = A["bin_n"][i]
            if n == 0:
                means.append({
                    "bin": (lo, hi),
                    "n": 0,
                    "f_ov": None, "r1": None, "q": None, "a": None
                })
            else:
                means.append({
                    "bin": (lo, hi),
                    "n": n,
                    "f_ov": A["f_ov"][i] / n,
                    "r1":   A["r1"][i]   / n,
                    "q":    A["q"][i]    / n,
                    "a":    A["a"][i]    / n,
                })
        out[cls] = means

    return out




In [12]:
# -------- Example usage --------
bins = [(0,10), (11,30), (31,100), (101, math.inf)]
res = analyze_target_difference_binned(
    train_loader,
    out_hw=(160,160),
    in_hw=(640,640),
    sigma=2.0,
    bins=bins,
    thr_mode="relative",  # recommended
    thr_rel=1e-3,
)
print(res["pos"])
print(res["neg"])

[{'bin': (0, 10), 'n': 120, 'f_ov': 0.0019557291190115695, 'r1': 0.01085728083593019, 'q': 0.9891427554912394, 'a': 1.0344426910536002}, {'bin': (11, 30), 'n': 276, 'f_ov': 0.01020649319938114, 'r1': 0.01767787709526893, 'q': 0.9823221452095973, 'a': 1.0699496217685223}, {'bin': (31, 100), 'n': 331, 'f_ov': 0.045078359988836005, 'r1': 0.026161716019331472, 'q': 0.9738382932333182, 'a': 1.0647514498029538}, {'bin': (101, inf), 'n': 73, 'f_ov': 0.18074806993954803, 'r1': 0.05107532028661619, 'q': 0.9489246756268394, 'a': 1.0356374097036245}]
[{'bin': (0, 10), 'n': 27, 'f_ov': 0.0025520832579651914, 'r1': 0.0122156860560828, 'q': 0.9877843527204446, 'a': 1.024953245820454}, {'bin': (11, 30), 'n': 118, 'f_ov': 0.00991525401911793, 'r1': 0.011822949432113244, 'q': 0.9881770708124727, 'a': 1.0372081563698157}, {'bin': (31, 100), 'n': 471, 'f_ov': 0.06311662211587095, 'r1': 0.02719875222996752, 'q': 0.9728012572311358, 'a': 1.0615769252066327}, {'bin': (101, inf), 'n': 184, 'f_ov': 0.25478812